In [1]:
# read new NICE framework 
import pandas as pd 

df = pd.read_excel('data/NICE Framework Components v2.0.0(2).xlsx')
print(df.head())
excel_file = pd.ExcelFile('data/NICE Framework Components v2.0.0(2).xlsx')
print('sheet names: ',excel_file.sheet_names[3:])
print(len(excel_file.sheet_names[3:]))

                                           Work Role  \
0  OVERSIGHT and GOVERNANCE (OG) – Provides leade...   
1        Communications Security (COMSEC) Management   
2                  Cybersecurity Policy and Planning   
3                 Cybersecurity Workforce Management   
4               Cybersecurity Curriculum Development   

                               Work Role Description Work Role ID  \
0                                                NaN          NaN   
1  Responsible for managing the Communications Se...   OG-WRL-001   
2  Responsible for developing and maintaining cyb...   OG-WRL-002   
3  Responsible for developing cybersecurity workf...   OG-WRL-003   
4  Responsible for developing, planning, coordina...   OG-WRL-004   

  OPM Code (Fed Use)                   Link to TKS List  Unnamed: 5  \
0                NaN                                NaN         NaN   
1                723  Click to view OG-WRL-001 TKS List         NaN   
2                752  Click to view

In [2]:
# read one of the sheets 
excel_file = pd.ExcelFile('data/NICE Framework Components v2.0.0(2).xlsx')
k_series_list = [] 
k_numbers_list = [] 

for i in range(3, len(excel_file.sheet_names)): 
    wrl = pd.read_excel('data/NICE Framework Components v2.0.0(2).xlsx', sheet_name=excel_file.sheet_names[i])
    #print(wrl.head())
    # find all elements in the first column that start with a K (the KDs). 
    
    # get the first column name 
    first_col = wrl.columns[0] 
    second_col = wrl.columns[1]
    KDs = wrl[wrl[first_col].astype(str).str.startswith('K')][second_col].tolist()
    k_series_list.append(KDs)
    KD_numbers = wrl[wrl[first_col].astype(str).str.startswith('K')][first_col].tolist()
    k_numbers_list.append(KD_numbers)

# concatenate all 
#combined_series = pd.concat(k_series_list, ignore_index=False)
#print(combined_series)
# find unique kds 
unique_strings = list(set(item for sublist in k_numbers_list for item in sublist))
#print(sorted(unique_strings))

unique_KDs = {}
for sublist1, sublist2 in zip(k_series_list, k_numbers_list): 
    for i in range(min(len(sublist1), len(sublist2))): 
        if sublist2[i].startswith('K'): 
            unique_KDs[sublist2[i]] = sublist1[i] 

#print(list(unique_KDs.values()))

In [3]:
# find the 2017 KDs in the unique 2025 KDs 
def extract_2017_KDs(dict):
    return [key for key in dict 
            if (isinstance(key, str) and 
                key[0] == 'K' and 
                int(key[1:]) <= 630)]

rep_KDs = extract_2017_KDs(unique_KDs)
print(rep_KDs)
# find the labels of each of these from the 2017 KDs 
# read Sara and Valtteri's annotations
from datasets import load_from_disk
ds = load_from_disk("data/train.hf")
repeated_KDs = [i for i, _ in enumerate(ds) if ds['KSAT ID'][i] in rep_KDs]
print(repeated_KDs)
print('Sara and Valtteris label')
#for i in repeated_KDs:
 #   print([x for x in ds[i] if ds[i][str(x)] == 1])

['K0018', 'K0498', 'K0476', 'K0092', 'K0055', 'K0068', 'K0176', 'K0159', 'K0064', 'K0480']
[17, 54, 63, 67, 90, 155, 170, 437, 441, 458]
Sara and Valtteris label


In [23]:
# find average length of KDs in 2017 and 2024 
import numpy as np 
lengths_2017 = [len(ele['Statement Description']) for i, ele in enumerate(ds)]
print(np.mean(lengths_2017))
print(len(ds))

lengths_2025 = [len(ele) for i, ele in enumerate(list(unique_KDs.values()))]
print(np.mean(lengths_2025))
print(ds['Statement Description'][0])
print(len(ds['Statement Description'][0]))
print('number of KDs: ',len(unique_KDs))

84.73090277777777
576
51.83216783216783
Knowledge of computer networking concepts and protocols, and network security methodologies
91
number of KDs:  572


In [42]:
print(ds[0])
print([x for x in ds[0] if ds[0][str(x)] == 1])

{'KSAT ID': 'K0001', '0': 0, '1': 0, '2': 0, '3': 0, '4': 1, '5': 0, '6': 0, '7': 0, '8': 0, 'Statement Description': 'Knowledge of computer networking concepts and protocols, and network security methodologies'}
['4']


In [4]:
# label all unique 2025 KDs
from datasets import Dataset 
df = pd.DataFrame({"Statement Description": unique_KDs})
KD2025 = Dataset.from_pandas(df, split='train')
print(KD2025)

# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
import torch
import numpy as np
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")

# load model from a working checkpoint 
model_name='BERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 
# misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()

Dataset({
    features: ['Statement Description', '__index_level_0__'],
    num_rows: 572
})


/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.2, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.2, inplace=False)


In [5]:
from transformers import AutoTokenizer
import datasets 
import re 
def clean_text(text): 
    special_chars = r'[\_/-]'
    cleaned_text = re.sub(special_chars,' ', text)                # replace - or _ or / with space 
    cleaned_text = re.sub(r'[^a-zA-Z0-9 -/()]', '', cleaned_text) # remove any non-alphanumeric /space character
    return cleaned_text.lower()  
    
def preprocess8(example):
    # preprocess and tokenize data 
    text = f"{example['Statement Description']}"       # our text to be labeled is the description of the KD
    text = text[13:]                                   # remove "Knowledge of" since it's the same for every KD
    text = clean_text(text)                            # remove special characters
    #one_hot = list(example.values())[1:9]              # read one-hot encoding 
    example = tokenizer(text, return_tensors='pt')                          # tokenize text 
    #example['labels'] = torch.tensor(one_hot).float()
    return example 
KD2025 = KD2025.map(preprocess8)
print(KD2025)

Parameter 'function'=<function preprocess8 at 0x7f84325eb550> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/572 [00:00<?, ? examples/s]

Dataset({
    features: ['Statement Description', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 572
})


In [6]:
print( KD2025[i] )

{'Statement Description': 'Knowledge of requirements analysis principles and practices', '__index_level_0__': 'K0690', 'input_ids': [[101, 5918, 4106, 6481, 1998, 6078, 102]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1]]}


In [7]:
llm_labels = [] 
tokenized_course = {}
KD2025_labels = {}
for i in range(len(KD2025)): 
    #print(course[str(i)])
    #tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
    row = KD2025[i] 
    print('KD: ',row['Statement Description'])
    print(row['__index_level_0__'])
    if row['__index_level_0__'] in rep_KDs:
        # take old labels if the KD is repeated from 2017 version
        print('Sara and Valtteris labels: ',[int(x) for x in ds[i] if ds[i][str(x)] == 1])
        KD2025_labels[row['__index_level_0__']] = [int(x) for x in ds[i] if ds[i][str(x)] == 1]
    else: 
        # use CurricuLLM to label any new KDs 
        with torch.no_grad(): 
            predictions= model(input_ids=torch.tensor(row['input_ids']), 
                               attention_mask=torch.tensor(row['attention_mask']))#**KD2025[str(i)])
            prediction = sigmoid(predictions.logits).detach().cpu().numpy()
            predictions = (prediction > 0.5).astype(int).reshape(-1)
            if sum(predictions) == 0: 
                    print('empty')
                    out = np.argmax(prediction)
                    prediction_zeros= np.zeros(predictions.shape)
                    prediction_zeros[out] = 1 
                    predictions = prediction_zeros.astype(int)
            llm_labels.append(predictions)
            
            print([i for i,x in enumerate(predictions) if x >0])
            KD2025_labels[row['__index_level_0__']] = [i for i,x in enumerate(predictions) if x >0]
# combine all labels of the course. 
#llm_labels2 = np.array(llm_labels)

KD:  Knowledge of encryption algorithms
K0018
Sara and Valtteris labels:  [4]
KD:  Knowledge of microprocessors
K0055
Sara and Valtteris labels:  [7]
KD:  Knowledge of performance tuning tools and techniques
K0064
Sara and Valtteris labels:  [8]
KD:  Knowledge of programming language structures and logic
K0068
Sara and Valtteris labels:  [2, 5, 8]
KD:  Knowledge of technology integration processes
K0092
Sara and Valtteris labels:  [0, 3, 4, 5]
KD:  Knowledge of Voice over IP (VoIP)
K0159
Sara and Valtteris labels:  [4, 7]
KD:  Knowledge of Extensible Markup Language (XML) schemas
K0176
Sara and Valtteris labels:  [5]
KD:  Knowledge of language processing tools and techniques
K0476
Sara and Valtteris labels:  [7]
KD:  Knowledge of malware
K0480
Sara and Valtteris labels:  [2]
KD:  Knowledge of operational planning processes
K0498
Sara and Valtteris labels:  [4]
KD:  Knowledge of decryption
K0635
[5]
KD:  Knowledge of decryption tools and techniques
K0636
[5]
KD:  Knowledge of data repos

In [ ]:
### For figure 3: Look at 3 specific KDs. 


In [10]:
# gather labels for each workrole 
for wr in range(0,42): 
    print(wr)
    wr_KD = [KD2025_labels[i] for i in k_numbers_list[wr]] # gather all assigned KAs to the KDs for this workrole
    # analyze the result 
    all_numbers = np.array([num for sublist in wr_KD for num in sublist]) 
    counts = np.bincount(all_numbers, minlength=9)
    prob_vector = counts / len(all_numbers)
    print(prob_vector)

0
[0.09589041 0.12328767 0.05479452 0.05479452 0.05479452 0.04109589
 0.08219178 0.38356164 0.10958904]
1
[0.14583333 0.0625     0.02083333 0.         0.08333333 0.02083333
 0.0625     0.4375     0.16666667]
2
[0.1884058  0.04347826 0.02898551 0.         0.07246377 0.
 0.04347826 0.50724638 0.11594203]
3
[0.33333333 0.06060606 0.         0.         0.10606061 0.01515152
 0.07575758 0.28787879 0.12121212]
4
[0.26436782 0.08045977 0.         0.         0.17241379 0.03448276
 0.08045977 0.26436782 0.10344828]
5
[0.16949153 0.11864407 0.01694915 0.         0.05084746 0.
 0.11864407 0.3559322  0.16949153]
6
[0.08474576 0.05084746 0.01694915 0.         0.15254237 0.03389831
 0.05084746 0.45762712 0.15254237]
7
[0.09638554 0.10843373 0.01204819 0.01204819 0.10843373 0.03614458
 0.13253012 0.31325301 0.18072289]
8
[0.17808219 0.04109589 0.04109589 0.02739726 0.06849315 0.02739726
 0.04109589 0.47945205 0.09589041]
9
[0.15625  0.046875 0.046875 0.03125  0.078125 0.03125  0.046875 0.453125
 0.10

In [8]:
# gather labels for each workrole 
wr = 9  # look at the first workrole
wr_KD = [KD2025_labels[i] for i in k_numbers_list[wr]] # gather all assigned KAs to the KDs for this workrole
print(wr_KD)
# analyze the result 
all_numbers = np.array([num for sublist in wr_KD for num in sublist]) 
counts = np.bincount(all_numbers, minlength=9)
prob_vector = counts / len(all_numbers)
print(counts)
print(prob_vector)

[[4], [2, 3], [2, 3], [7], [4], [7], [7, 8], [7], [1, 6, 7, 8], [1, 6, 7, 8], [7, 8], [1, 6, 7, 8], [7, 8], [7, 8], [7], [5], [7], [7], [5], [7], [0], [7], [7], [0], [7], [7], [7], [7], [0], [2], [7], [7], [0], [7], [7], [7], [4], [0], [7], [0], [0], [0], [7], [0], [4], [4], [7], [7], [0]]
[10  3  3  2  5  2  3 29  7]
[0.15625  0.046875 0.046875 0.03125  0.078125 0.03125  0.046875 0.453125
 0.109375]


In [15]:
print(excel_file.sheet_names[1:])

['v2.0.0 TKS Statements', 'Competency Area Catalog', 'OG-WRL-001', 'OG-WRL-002', 'OG-WRL-003', 'OG-WRL-004', 'OG-WRL-005', 'OG-WRL-006', 'OG-WRL-007', 'OG-WRL-008', 'OG-WRL-009', 'OG-WRL-010', 'OG-WRL-011', 'OG-WRL-012', 'OG-WRL-013', 'OG-WRL-014', 'OG-WRL-015', 'OG-WRL-016', 'DD-WRL-001', 'DD-WRL-002', 'DD-WRL-003', 'DD-WRL-004', 'DD-WRL-005', 'DD-WRL-006', 'DD-WRL-007', 'DD-WRL-008', 'DD-WRL-009', 'IO-WRL-001', 'IO-WRL-002', 'IO-WRL-003', 'IO-WRL-004', 'IO-WRL-005', 'IO-WRL-006', 'IO-WRL-007', 'PD-WRL-001', 'PD-WRL-002', 'PD-WRL-003', 'PD-WRL-004', 'PD-WRL-005', 'PD-WRL-006', 'PD-WRL-007', 'IN-WRL-001', 'IN-WRL-002', 'NF-COM-007']


In [12]:
from collections import defaultdict 

def compute_wr_prob(KD2025_labels, k_numbers_list, wr): 
    wr_KD = [KD2025_labels[i] for i in k_numbers_list[wr]] # gather all assigned KAs to the KDs for this workrole
   # print(wr_KD)
    # analyze the result 
    all_numbers = np.array([num for sublist in wr_KD for num in sublist]) 
    counts = np.bincount(all_numbers, minlength=9)
    prob_vector = counts / len(all_numbers)
    #print(counts)
    #print(prob_vector)
    return prob_vector 

def get_category_indices(string_list): 
    category_indices = defaultdict(list)
    for idx, string in enumerate(string_list): 
        category = string.split('-')[0]
        category_indices[category].append(idx)
    return dict(category_indices)

indices_by_category = get_category_indices(excel_file.sheet_names[3:])
print(indices_by_category)
for category, indices in indices_by_category.items(): 
    print('Category: ',category)
    category_vector = [] 
    for idx in indices: 
        wr_vector = compute_wr_prob(KD2025_labels, k_numbers_list, idx)
        category_vector.append(wr_vector)

    category_vector = np.array(category_vector)
    mean_category = np.sum(category_vector, axis=0) / np.sum(category_vector)
    print('KA distribution: ',100*mean_category)

{'OG': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], 'DD': [16, 17, 18, 19, 20, 21, 22, 23, 24], 'IO': [25, 26, 27, 28, 29, 30, 31], 'PD': [32, 33, 34, 35, 36, 37, 38], 'IN': [39, 40], 'NF': [41]}
Category:  OG
KA distribution:  [16.49953672  7.49485815  2.95590358  1.39889152  9.24467063  2.70034185
  6.74740551 40.59281001 12.36558204]
Category:  DD
KA distribution:  [13.54284954  6.34556057  9.92689892  3.84334969 11.36573028  9.0834038
  5.30459689 32.3545491   8.2330612 ]
Category:  IO
KA distribution:  [20.12738482 10.87022977  3.57729859  1.83533596 12.2396893   4.9124129
  8.07336324 28.56651931  9.79776611]
Category:  PD
KA distribution:  [15.29913652 11.14544388  2.67366917  0.88998005 15.86268173  5.10684211
  6.47231261 33.15451926  9.39541466]
Category:  IN
KA distribution:  [12.46130031 21.10423117  3.36687307  1.75438596  9.23632611  7.03044376
  4.69556244 28.17337461 12.17750258]
Category:  NF
KA distribution:  [19.67213115  1.63934426  6.55737705  3.27868852

In [36]:
# combine Protect and Defend and Investigate old 2017 
arr1 = 4*np.array([12, 7, 6, 1.5, 33, 14, 1.5, 17, 8])
arr2 = 3*np.array([16, 31, 5, 2, 12, 6, 0, 16, 12])
arr = arr1 + arr2
print(100*arr / sum(arr))


[13.71428571 17.28571429  5.57142857  1.71428571 24.         10.57142857
  0.85714286 16.57142857  9.71428571]


In [51]:
WR_OG = 16 # Number of job roles in OG
WR_DD = 9  # Number of job roles in DD 
WR_IO = 7  # Number of job roles in IO
WR_PD = 7  # Number of job roles in PD
WR_IN = 2  # Number of job roles in IN

X_DD = 19 # Percentage of job demands for job category 1
X_IO = 14 # Percentage of job demands for job category 2
X_OG = 17 # Percentage of job demands for job category 3
X_PD = 17 # Percentage of job demands for job category 4
X_AN = (4*13 + 3*10)/7 # Percentage of job demands for job category 5
total = X_DD + X_IO + X_OG + X_PD + X_AN 
X_DD = X_DD / total * 100 
X_IO = X_IO / total * 100 
X_OG = X_OG / total * 100 
X_PD = X_PD / total * 100 
X_IN = X_AN / total * 100 
print(X_DD + X_IO + X_OG + X_PD + X_IN) 

100.0


In [75]:
demand = [
    12818, 35998, 1978, 21381, 11834, 35071, 3122, 59953, 3295, 28465, 88569, 57391, 138587, 63479, 3265, 128548,
    80335, 25621, 105906, 65408, 69469, 64043, 10035, 835, 0, # operational technology cybersecurity engineering
    160418, 33164, 33856, 87882, 83458, 84336, 67116, 
    43286, 13388, 68146, 78728, 43547, 78772, 97897, 
    9367, 14425
]
print('Systems Authorization: ',sum(demand[:16]))
print('Design & Development: ',sum(demand[16:25]))
print('Implementation & Operation: ',sum(demand[25:32]))
print('Protection & Defense: ',sum(demand[32:39]))
print('Investigation: ',sum(demand[39:]))

print(sum(demand))
print(len(demand))
WR_number = [len(l) for l in k_numbers_list][:41] # number of KDs in a work role.
print(len(WR_number))
WR_weight_per_KD = [demand[i]/ WR_number[i] for i in range(len(WR_number))] 
WR_weight_per_KD = [WR_weight_per_KD[i] / sum(WR_weight_per_KD) for i in range(len(WR_number))]
print('weight each KD in a specified work role gets: ',WR_weight_per_KD)

Systems Authorization:  693754
Design & Development:  421652
Implementation & Operation:  550230
Protection & Defense:  423764
Investigation:  23792
2113192
41
41
weight each KD in a specified work role gets:  [0.007795493484378369, 0.03283923089998274, 0.0011361248240569122, 0.013533977658433434, 0.005319563481382402, 0.025899559680810324, 0.0021518594746444976, 0.038740320032239414, 0.001762061032952533, 0.01801808493743547, 0.0528289427477485, 0.013485386018867824, 0.05731321249440557, 0.017737835360773332, 0.003375634778049784, 0.10492409916548973, 0.0205926837275863, 0.008928933563165358, 0.03690830326453263, 0.017048156760344393, 0.025054521784572865, 0.0233693301242024, 0.005187518376375128, 0.0003547783724858801, 0.0, 0.07538808016499934, 0.01804619195280778, 0.027634115671553192, 0.03284092351127192, 0.03276682815125658, 0.029725135873758173, 0.04003282549715914, 0.010740659871558686, 0.004417548963070331, 0.03843007226812985, 0.04787982610689189, 0.018009037388022536, 0.03214

In [91]:
# now loop over all workroles and gather the weight for all KDs 
KD_weights = dict.fromkeys(KD2025_labels.keys(), 0) 
for i, workrole in enumerate(k_numbers_list[:41]): 
    for kd in workrole: 
        KD_weights[kd] += WR_weight_per_KD[i] 

print(len(KD_weights))
print(len(KD2025_labels))
print(KD_weights)

572
572
{'K0018': 0.1770600704444581, 'K0055': 0.06101017061213309, 'K0064': 0.03276682815125658, 'K0068': 0.2177714187063139, 'K0092': 0.03833051908835963, 'K0159': 0.03284092351127192, 'K0176': 0.0003547783724858801, 'K0476': 0.013485386018867824, 'K0480': 0.032147840259918306, 'K0498': 0.11687690300307019, 'K0635': 0.02242658635109287, 'K0636': 0.027237487074638602, 'K0637': 0.02242658635109287, 'K0638': 0.005319563481382402, 'K0639': 0.03690830326453263, 'K0640': 0.05844933731846248, 'K0641': 0.001762061032952533, 'K0642': 0.001762061032952533, 'K0643': 0.018853541139815834, 'K0644': 0.13218074772532915, 'K0645': 0.11153997368065513, 'K0646': 0.02952161729075166, 'K0647': 0.07538808016499934, 'K0648': 0.0011361248240569122, 'K0649': 0.0011361248240569122, 'K0650': 0.0726090887181365, 'K0651': 0.0726090887181365, 'K0652': 0.0011361248240569122, 'K0653': 0.20806673066283843, 'K0654': 0.018853541139815834, 'K0655': 0.04563322627878613, 'K0656': 0.018009037388022536, 'K0657': 0.0180090

In [96]:
# Take KD2025_labels dictionary and translate KD weights back to KAs
KA_weights = {'0': 0, '1': 0, '2': 0, '3': 0, '4': 0, '5': 0,'6': 0, '7': 0,'8': 0} 
KD_weights_list = [elem for elem in KD_weights.values()]
for i, KA in enumerate(KD2025_labels.values()): 
    for label in KA: 
        KA_weights[str(label)] += KD_weights_list[i]

KA_weights_final = [elem for elem in KA_weights.values()]
KA_weights_final = [elem / sum(KA_weights_final)*100 for elem in KA_weights_final]
print(KA_weights_final)
print(sum(KA_weights_final))

[16.266082055540878, 8.89288636346437, 5.277423285531892, 1.8547398305848455, 11.699787328186522, 5.5394732191161795, 6.8977797549608795, 33.350123394399034, 10.2217047682154]
100.0


### Old code 

In [50]:
#print(k_series_list)
#print(k_numbers_list) 
print(len(k_numbers_list))
matrix_column = []

WR_number = [len(l) for l in k_numbers_list] # number of KDs in a work role.
print(WR_number)
a = float(X_OG)/WR_OG
for i in range(0,WR_OG):
    matrix_column.append(float(a)/WR_number[i])

a = float(X_DD)/WR_DD
for i in range(0,WR_DD):
    matrix_column.append(float(a)/WR_number[i+WR_OG])

WR_OG = 16 # Number of job roles in OG
WR_DD = 9  # Number of job roles in DD 
WR_IO = 7  # Number of job roles in IO
WR_PD = 7  # Number of job roles in PD
WR_IN = 2  # Number of job roles in IN

a = float(X_IO)/WR_IO
for i in range(0,WR_IO):
    matrix_column.append(float(a)/WR_number[i+WR_OG+WR_DD])

a = float(X_PD)/WR_PD
for i in range(0,WR_PD):
    matrix_column.append(float(a)/WR_number[i+WR_OG+WR_DD+WR_IO])

a = float(X_IN)/WR_IN
for i in range(0,WR_IN):
    matrix_column.append(float(a)/WR_number[i+WR_OG+WR_DD+WR_IO+WR_PD])

print(matrix_column)

42
[51, 34, 54, 49, 69, 42, 45, 48, 58, 49, 52, 132, 75, 111, 30, 38, 121, 89, 89, 119, 86, 85, 60, 73, 47, 66, 57, 38, 83, 79, 88, 52, 125, 94, 55, 51, 75, 76, 64, 54, 93, 56]
[0.026467029643073205, 0.039700544464609806, 0.024996639107346914, 0.027547316567280274, 0.01956258712748889, 0.03213853599516032, 0.029995966928816297, 0.02812121899576528, 0.023272732962012644, 0.027547316567280274, 0.025958048303783336, 0.01022589781664192, 0.017997580157289778, 0.012160527133303905, 0.044993950393224444, 0.03552153978412456, 0.022165225926981412, 0.030134745361401694, 0.030134745361401694, 0.022537750732476897, 0.03118595740889245, 0.031552851025467656, 0.04469987228607918, 0.036739621057051385, 0.05706366674818619, 0.038497497662651925, 0.044576049925175915, 0.06686407488776387, 0.030612468020903942, 0.0321624664017092, 0.028873123246988944, 0.048862208571827444, 0.024682395644283126, 0.032822334633355224, 0.0560963537370071, 0.0604960677555959, 0.04113732607380521, 0.04059604546757093, 0.0

In [ ]:
matrix_row = []
for i in range(1,631):
    KD_repeat = []
    for j in range(0,len(WR_all)):
        if i in WR_all[j]:
            KD_repeat.append(j)
    matrix_row.append(KD_repeat)
#print('matrix_row',len(matrix_row))
#print('matrix_row',matrix_row)

weight_KD = []

for i in range(0,len(matrix_row)): #len(matrix_row) is 630 in our case
    row_sum = 0
    for j in range(0,len(matrix_column)): #len(matrix_column) is 52 in our case
        if j in matrix_row[i]:
            row_sum = matrix_column[j] + row_sum
    weight_KD. append(row_sum)

print(weight_KD)
print(sum(weight_KD))
#print(weight_KD[373])

In [83]:
print('sheet names: ',excel_file.sheet_names[3:])

sheet names:  ['OG-WRL-001', 'OG-WRL-002', 'OG-WRL-003', 'OG-WRL-004', 'OG-WRL-005', 'OG-WRL-006', 'OG-WRL-007', 'OG-WRL-008', 'OG-WRL-009', 'OG-WRL-010', 'OG-WRL-011', 'OG-WRL-012', 'OG-WRL-013', 'OG-WRL-014', 'OG-WRL-015', 'OG-WRL-016', 'DD-WRL-001', 'DD-WRL-002', 'DD-WRL-003', 'DD-WRL-004', 'DD-WRL-005', 'DD-WRL-006', 'DD-WRL-007', 'DD-WRL-008', 'DD-WRL-009', 'IO-WRL-001', 'IO-WRL-002', 'IO-WRL-003', 'IO-WRL-004', 'IO-WRL-005', 'IO-WRL-006', 'IO-WRL-007', 'PD-WRL-001', 'PD-WRL-002', 'PD-WRL-003', 'PD-WRL-004', 'PD-WRL-005', 'PD-WRL-006', 'PD-WRL-007', 'IN-WRL-001', 'IN-WRL-002', 'NF-COM-007']


In [6]:
# Executing the following code results in assigning weights to
# Knowldege Descriptions (KD) in different Work Roles (WR) that
# are defined by NIST and published as NICE Framework.

# The following persentahges are base on the Prof. Martti Lehto's report.

X_SP = 19 # Percentage of job demands for job category 1
X_OM = 14 # Percentage of job demands for job category 2
X_OV = 17 # Percentage of job demands for job category 3
X_PR = 17 # Percentage of job demands for job category 4
X_AN = 13 # Percentage of job demands for job category 5
X_CO = 10 # Percentage of job demands for job category 6
X_IN = 10 # Percentage of job demands for job category 7

WR_SP = 11 # number of job roles in SP
WR_OM = 7 # number of job roles in OM
WR_OV = 14 # number of job roles in OV
WR_PR = 4 # number of job roles in PR
WR_AN = 7 # number of job roles in AN
WR_CO = 6 # number of job roles in CO
WR_IN = 3 # number of job roles in IN

### Arthur's code 
print('Divide the percentage allocated to job category X by the number of work roles in said category')
print('weight of Protect and defend', X_PR/WR_PR, '%')

52
weight of Protect and defend 4.25 %


This first step does not make much sense to me. 
It feels more logical to look for each of the work roles and see how often these occur in job advertisements, 
instead of 

In [ ]:
# new framework: 
WR_OG = 16 # Oversight and Governance 
WR_DD = 9  # Design and Development 
WR_IO = 7  # Implementation and Operation
WR_PD = 7  # Protection and Defense 
WR_IN = 2  # Investigation 

In [ ]:
### The following piece of code converts the id's of KDs to an integer########

##WR = '' # Insert the id's for KDs here
##
##WR_new= []
##i = 0
##while i <= len(WR)-1:
##    if WR[i] == 'K':
##        WR_new.append(WR[i+2:i+5])
##    i = i+1
##WR_new2=[]
##for i in WR_new:
##    if i[0]=='0':
##        j = i[1:]
##        if j[0] == '0':
##            j = j[1:]
##        WR_new2.append(j)
##    else:
##        WR_new2.append(i)
##
##WR_new1 = []
##for i in WR_new2:
##    WR_new1.append(int(i))
##print WR_new1

WR_all = [[1, 2, 3, 4, 5, 6, 13, 19, 27, 28, 37, 38, 40, 44, 48,
       49, 54, 59, 70, 84, 89, 101, 126, 146, 168, 169, 170, 179, 199,
      203, 260, 261, 262, 267, 295, 322, 342, 622, 624],[1, 2, 3, 4, 5,
        6, 7, 8, 9, 10, 11, 13, 18, 19, 21, 24, 26, 27, 28, 29, 37, 38, 40, 44, 48, 49, 54,
        56, 59, 70, 84, 89, 98, 100, 101, 126, 146, 168, 169,
        170, 179, 199, 203, 260, 261, 262, 267, 287, 322, 342, 622, 624],
          [1, 2, 3, 4, 5, 6, 14, 16, 27, 28, 39, 44, 50, 51, 60, 66, 68, 70,
        73, 79, 80, 81, 82, 84, 86, 105, 139, 140, 152, 153, 154, 170, 179,
        199, 202, 260, 261, 262, 263, 322, 332, 342, 343, 624],
          [1, 2, 3, 4, 5, 6, 14, 16, 27, 28, 39, 44, 50, 51, 60, 66, 68, 70,
        73, 79, 80, 81, 82, 84, 86, 105, 139, 140, 152, 153, 154, 170, 178,
        179, 199, 202, 260, 261, 262, 263, 322, 342, 343, 624],
          [1, 2, 3, 4, 5, 6, 24, 27, 28, 30, 35, 37, 43, 44, 52, 56, 60, 61, 63,
        74, 75, 82, 91, 93, 102, 170, 179, 180, 198, 200, 203, 207, 211, 212,
        214, 227, 240, 264, 275, 286, 287, 291, 293, 299, 322, 323, 325, 326,
        332, 333, 487, 516], [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 15, 18, 19, 24, 26, 27, 30,
        35, 36, 37, 43, 44, 52, 55, 56, 57, 59, 60, 61, 63, 71, 74, 82, 91, 92,
        93, 102, 170, 180, 198, 200, 202, 211, 212, 214, 227, 240, 260, 261,
        262, 264, 275, 277, 286, 287, 291, 293, 320, 322, 323, 325, 326, 332,
        333, 336, 565], [1, 2, 3, 4, 5, 6, 9, 19, 59, 90, 126, 169, 170, 171, 172, 174, 175, 176,
        179, 202, 209, 267, 268, 269, 271, 272, 288, 296, 310, 314, 321, 342,
        499], [1, 2, 3, 4, 5, 6, 8, 12, 18, 19, 32, 35, 38, 43, 44, 45, 47, 55, 56,
        59, 60, 61, 63, 66, 67, 73, 74, 86, 87, 90, 91, 93, 101, 102, 126, 163,
        164, 168, 169, 170, 180, 200, 267, 287, 325, 332, 333, 622],
          [1, 2, 3, 4, 5, 6, 27, 28, 37, 44, 57, 88, 91, 102, 139, 126, 169,
        170, 179, 199, 203, 212, 250, 260, 261, 262, 287, 332],
          [1, 2, 3, 4, 5, 6, 15, 18, 24, 27, 28, 30, 32, 35, 36, 44, 45, 49, 50,
         52, 55, 56, 60, 61, 63, 65, 66, 67, 73, 81, 82, 84, 86, 87, 90, 91,
         93, 102, 126, 139, 169, 170, 179, 180, 200, 203, 260, 261, 262, 276,
         287, 297, 308, 322, 325, 332, 333, 336],
          [1, 2, 3, 4, 5, 6, 15, 18, 24, 27, 28, 30, 32, 35, 36, 44, 45, 49, 50,
         52, 55, 56, 60, 61, 63, 65, 66, 67, 73, 81, 82, 84, 86, 87, 90, 91,
         93, 102, 126, 139, 169, 170, 179, 180, 200, 203, 207, 212, 227, 260,
         261, 262, 276, 287, 297, 308, 322, 325, 332, 333, 336],
          [1, 2, 3, 4, 5, 6, 20, 21, 22, 23, 25, 31, 56, 60, 65, 69, 83, 97,
         197, 260, 261, 262, 277, 278, 287, 420],
          [1, 2, 3, 4, 5, 6, 15, 16, 20, 22, 23, 25, 31, 51, 52, 56, 60, 65,
         68, 69, 83, 95, 129, 139, 140, 193, 197, 229, 236, 238, 325, 420],
          [1, 2, 3, 4, 5, 6, 13, 94, 95, 96, 146, 194, 195, 228, 260, 261, 262,
         283, 287, 315, 338, 420],
          [1, 2, 3, 4, 5, 6, 53, 88, 109, 114, 116, 194, 224, 237, 242, 247,
         260, 261, 262, 287, 292, 294, 302, 317, 330],
          [1, 2, 3, 4, 5, 6, 10, 11, 29, 38, 49, 50, 53, 61, 71, 76, 93, 104, 108, 111,
         113, 135, 136, 137, 138, 159, 160, 179, 180, 200, 201, 203, 260, 261, 262,
         274, 287, 332, 622],
          [1, 2, 3, 4, 5, 6, 49, 50, 53, 64, 77, 88, 100, 103, 104, 117, 130, 158, 167,
         179, 260, 261, 262, 274, 280, 289, 318, 332, 346],
          [1, 2, 3, 4, 5, 6, 15, 18, 19, 24, 35, 36, 40, 44, 49, 52, 56, 60, 61, 63, 75,
         82, 93, 102, 179, 180, 200, 203, 227, 260, 261, 262, 263, 266, 267, 275, 276,
         281, 284, 285, 287, 290, 297, 322, 333, 339],
          [1, 2, 3, 4, 5, 6, 17, 59, 107, 157, 261, 262, 267, 312, 316, 341, 615],
          [1, 2, 3, 4, 5, 6, 8, 66, 168, 612, 613, 614, 615],
          [1, 2, 3, 4, 5, 6, 59, 124, 146, 147, 204, 208, 213, 216, 217, 220, 243,
         239, 245, 246, 250, 252, 287, 628], [1, 2, 3, 4, 5, 6, 7, 59, 115, 124, 130, 146, 147, 204, 208, 213, 215, 216,
         217, 218, 220, 226, 239, 245, 246, 250, 252, 287, 313, 319, 628],
          [1, 2, 3, 4, 5, 6, 8, 18, 21, 26, 33, 38, 40, 42, 43, 46, 48, 53, 54, 58,
         59, 61, 70, 72, 76, 77, 87, 90, 92, 101, 106, 121, 126, 149, 150, 151, 163,
         167, 168, 169, 170, 179, 180, 199, 260, 261, 262, 267, 287, 332, 342, 622, 624],
          [1, 2, 3, 4, 5, 6, 18, 26, 38, 42, 90, 101, 121, 126, 163, 267, 285, 287, 622],
          [1, 2, 3, 4, 5, 6, 72, 101, 127, 146, 147, 168, 169, 204, 215, 233, 234, 241,
         243, 309, 311, 313, 335], [1, 2, 3, 4, 5, 6, 70, 127, 146, 168, 234, 248, 309, 311, 313, 335, 624],
          [1, 2, 3, 4, 5, 6, 9, 70, 106, 314, 296, 147, 624, 628],
          [1, 2, 3, 4, 5, 6, 47, 48, 72, 90, 101, 120, 126, 146, 148, 154, 164, 165,
         169, 194, 196, 198, 200, 235, 257, 270],
          [1, 2, 3, 4, 5, 6, 12, 43, 47, 48, 59, 72, 90, 101, 120, 126, 146, 148, 154,
         164, 165, 169, 194, 196, 198, 200, 235, 257, 270],
          [1, 2, 3, 4, 5, 6, 43, 48, 59, 72, 90, 120, 126, 148, 150, 154, 164, 165,
         169, 194, 196, 198, 200, 235, 249, 257, 270],
          [1, 2, 3, 4, 5, 6, 48, 72, 120, 126, 146, 154, 165, 169, 235, 257, 270],
          [1, 2, 3, 4, 5, 6, 43, 47, 48, 72, 90, 120, 126, 148, 154, 165, 169, 198,
         200, 235, 257, 270], [1, 2, 3, 4, 5, 6, 7, 13, 15, 18, 19, 24, 33, 40, 42, 44, 46, 49, 56, 58,
         59, 60, 61, 65, 70, 74, 75, 93, 98, 104, 106, 107, 110, 111, 112, 113, 116,
         139, 142, 143, 157, 160, 161, 162, 167, 168, 177, 179, 180, 190, 191, 192,
         203, 221, 222, 260, 261, 262, 290, 297, 300, 301, 303, 318, 322, 324, 332,
         339, 342, 624],
          [1, 2, 3, 4, 5, 6, 21, 33, 42, 44, 58, 61, 62, 104, 106, 135, 157, 179, 205,
         258, 274, 324, 332, 334],
          [1, 2, 3, 4, 5, 6, 21, 26, 33, 34, 41, 42, 46, 58, 62, 70, 106, 157, 161,
         162, 167, 177, 179, 221, 230, 259, 287, 332, 565, 624],
          [1, 2, 3, 4, 5, 6, 9, 19, 21, 33, 44, 56, 61, 68, 70, 89, 106, 139, 161,
         162, 167, 177, 179, 203, 206, 210, 224, 265, 287, 301, 308, 332, 342, 344, 624],
          [1, 2, 3, 4, 5, 6, 36, 58, 108, 109, 177, 349, 362, 377, 392, 395, 405, 409,
         415, 417, 427, 431, 436, 437, 440, 444, 445, 446, 449, 458, 460, 464, 469,
         471, 480, 499, 511, 516, 556, 560, 561, 565, 603, 604, 610, 612, 614],
          [1, 2, 3, 4, 5, 6, 108, 109, 131, 142, 143, 177, 224, 349, 362, 417, 444,
         471, 560, 351, 354, 368, 371, 376, 379, 388, 393, 394, 397, 418, 430, 443,
         447, 451, 470, 473, 484, 487, 489, 509, 510, 523, 529, 535, 544, 557, 559, 608],
          [1, 2, 3, 4, 5, 6, 36, 58, 108, 109, 177, 221, 349, 362, 444, 471, 560, 377,
         392, 395, 405, 409, 427, 431, 436, 437, 440, 445, 446, 449, 458, 460, 464, 469,
         480, 511, 516, 556, 561, 565, 603, 604, 610, 612, 614, 357, 410, 457, 465, 507,
         533, 542, 549, 551, 577, 598],
          [1, 2, 3, 4, 5, 6, 36, 58, 108, 109, 177, 349, 362, 377, 392, 395, 405,
         409, 410, 414, 417, 427, 431, 436, 437, 440, 444, 445, 446, 449, 457,
         460, 464, 465, 469, 471, 480, 507, 511, 516, 549, 551, 556, 560, 561,
         565, 598, 603, 604, 610, 612, 614],
          [1, 2, 3, 4, 5, 6, 36, 58, 108, 109, 177, 142, 349, 362, 444, 471,
         560, 392, 395, 409, 427, 431, 436, 437, 440, 445, 446, 449, 460, 464,
         516, 556, 561, 565, 603, 604, 614, 457, 465, 507, 549, 551, 598, 417,
         458, 357, 533, 542, 351, 379, 473, 381, 402, 413, 426, 439, 461, 466,
         478, 479, 497, 499, 543, 546, 547, 555],
          [1, 2, 3, 4, 5, 6, 108, 109, 177, 349, 362, 444, 471, 392, 395, 431, 436,
         440, 445, 449, 516, 379, 473, 413, 439, 479, 547, 487, 544, 559, 389, 403,
         424, 442, 462, 472, 483, 499, 500, 520, 550, 567, 592, 599, 600],
          [1, 2, 3, 4, 5, 6, 108, 143, 177, 431, 449, 413, 487, 462, 520, 550, 567,
         599, 600, 417, 377, 356, 359, 391, 396, 398, 407, 416, 476, 488, 491, 493,
         499, 524, 532, 539, 540, 541, 545, 548, 564, 571, 574, 579, 596, 606, 607],
          [1, 2, 3, 4, 5, 6, 36, 58, 109, 177, 431, 449, 417, 579, 596, 444, 471, 392,
         395, 440, 445, 516, 560, 427, 446, 561, 565, 405, 480, 610, 612, 353, 361,
         364, 380, 382, 383, 386, 387, 390, 401, 404, 412, 419, 425, 435, 448, 453,
         454, 467, 474, 475, 477, 482, 492, 495, 496, 498, 503, 505, 513, 521, 522,
         526, 527, 552, 553, 554, 558, 562, 563, 569, 570, 580, 581, 583, 584, 587,
         588, 601, 605, 613],
          [1, 2, 3, 4, 5, 6, 36, 58, 109, 177, 353, 361, 364, 380, 382, 383, 384,
         386, 387, 390, 395, 401, 404, 412, 417, 419, 421, 425, 427, 431, 435, 444,
         445, 446, 448, 453, 454, 467, 474, 475, 477, 480, 482, 492, 495, 496, 498,
         505, 513, 516, 521, 526, 527, 552, 554, 558, 560, 561, 562, 563, 565, 568,
         569, 570, 579, 580, 581, 584, 587, 588, 596, 605, 610, 612],
          [1, 2, 3, 4, 5, 6, 36, 108, 109, 120, 431, 417, 444, 395, 445, 560, 427,
         446, 561, 565, 480, 610, 612, 435, 471, 392, 440, 405, 377, 349, 362,
         436, 379, 403, 460, 464, 556, 603, 614, 465, 507, 598, 511, 414, 577,
         347, 350, 352, 355, 358, 399, 400, 408, 411, 422, 432, 455, 456, 459,
         463, 494, 499, 501, 502, 504, 506, 508, 512, 514, 517, 518, 519, 525,
         538, 566, 572, 575, 578, 582, 585, 586, 589, 590, 591, 593, 594, 595, 599, 602],
          [1, 2, 3, 4, 5, 6, 36, 108, 109, 347, 349, 350, 352, 362, 377, 379, 392,
         395, 399, 400, 403, 408, 411, 414, 417, 422, 431, 432, 435, 436, 444,
         445, 446, 455, 464, 465, 471, 480, 494, 497, 499, 501, 502, 504, 506,
         507, 508, 511, 512, 514, 516, 518, 519, 525, 534, 538, 556, 560, 561,
         565, 566, 572, 576, 582, 585, 586, 589, 590, 593, 594, 597, 598, 599,
         603, 610, 612, 614],
          [1, 2, 3, 4, 5, 6, 108, 109, 431, 417, 444, 395, 435, 392, 377, 362,
         436, 379, 403, 465, 507, 598, 511, 414, 350, 400, 408, 411, 422, 432,
         455, 499, 501, 504, 506, 508, 512, 514, 538, 585, 599],
          [1, 2, 3, 4, 5, 6, 9, 21, 51, 109, 142, 224, 363, 372, 373, 375, 379,
         403, 406, 420, 423, 428, 427, 429, 430, 433, 438, 440, 452, 468, 481,
         485, 486, 480, 516, 528, 530, 531, 536, 560, 565, 573, 608, 609],
          [1, 2, 3, 4, 5, 6, 46, 70, 107, 110, 114, 118, 123, 125, 128, 144,
         155, 156, 168, 209, 231, 244, 251, 351, 624],
          [1, 2, 3, 4, 5, 6, 17, 21, 42, 60, 70, 77, 78, 107, 109, 117, 118,
         119, 122, 123, 125, 128, 131, 132, 133, 134, 145, 155, 156, 167,
         168, 179, 182, 183, 184, 185, 186, 187, 188, 189, 305, 624],
          [1, 2, 3, 4, 5, 6, 18, 21, 42, 60, 70, 77, 78, 109, 117, 118, 119,
         122, 123, 125, 128, 131, 132, 133, 134, 145, 155, 156, 167, 168, 179,
         182, 183, 184, 185, 186, 187, 188, 189, 224, 254, 255, 301, 304, 347, 624]]

   

In [1]:
##WR_number = []
##
##for i in WR_all:
##    WR_number.append(len(i))
##print WR_number

WR_number = [39, 52, 44, 44, 52, 69, 33, 48, 28, 58, 61, 26, 32, 22,
             25, 39, 29, 46, 17, 13, 24, 31, 53, 19, 23, 17, 14, 26,
             29, 27, 17, 22, 70, 24, 30, 35, 47, 48, 56, 52, 66, 45,
             47, 82, 74, 89, 77, 41, 44, 25, 42, 46] # number of KDs in a work role.

matrix_column = []

a = float(X_SP)/WR_SP
for i in range(0,WR_SP):
    matrix_column.append(float(a)/WR_number[i])

a = float(X_OM)/WR_OM
for i in range(0,WR_OM):
    matrix_column.append(float(a)/WR_number[i+WR_SP])

a = float(X_OV)/WR_OV
for i in range(0,WR_OV):
    matrix_column.append(float(a)/WR_number[i+WR_SP+WR_OM])

a = float(X_PR)/WR_PR
for i in range(0,WR_PR):
    matrix_column.append(float(a)/WR_number[i+WR_SP+WR_OM+WR_OV])

a = float(X_AN)/WR_AN
for i in range(0,WR_AN):
    matrix_column.append(float(a)/WR_number[i+WR_SP+WR_OV+WR_PR+WR_OM])

a = float(X_CO)/WR_CO
for i in range(0,WR_CO):
    matrix_column.append(float(a)/WR_number[i+WR_SP+WR_OV+WR_PR+WR_AN+WR_OM])

a = float(X_IN)/WR_IN
for i in range(0,WR_IN):
    matrix_column.append(float(a)/WR_number[i+WR_SP+WR_OV+WR_PR+WR_AN+WR_CO+WR_OM])
#print('matrix col',matrix_column)
#print('sum of matrix col', sum(matrix_column))

matrix_row = []
for i in range(1,631):
    KD_repeat = []
    for j in range(0,len(WR_all)):
        if i in WR_all[j]:
            KD_repeat.append(j)
    matrix_row.append(KD_repeat)
#print('matrix_row',len(matrix_row))
#print('matrix_row',matrix_row)

weight_KD = []

for i in range(0,len(matrix_row)): #len(matrix_row) is 630 in our case
    row_sum = 0
    for j in range(0,len(matrix_column)): #len(matrix_column) is 52 in our case
        if j in matrix_row[i]:
            row_sum = matrix_column[j] + row_sum
    weight_KD. append(row_sum)

print(weight_KD)
print(sum(weight_KD))
#print(weight_KD[373])



[2.9127967063984417, 2.9127967063984417, 2.9127967063984417, 2.9127967063984417, 2.9127967063984417, 2.9127967063984417, 0.15813451391992767, 0.21055221439757943, 0.3566333722742532, 0.10953177257525083, 0.10953177257525083, 0.10288970774353083, 0.25416214220562044, 0.07851239669421488, 0.24982199527232338, 0.1410123966942149, 0.1507936507936508, 0.4158082207389253, 0.4164863298756579, 0.13942307692307693, 0.7629371181411788, 0.13942307692307693, 0.13942307692307693, 0.2537555617058898, 0.13942307692307693, 0.2867372136088944, 0.3340527677936097, 0.3090198297171934, 0.0844988344988345, 0.11634623190525567, 0.13942307692307693, 0.0940813590969046, 0.5238039083557952, 0.14166666666666666, 0.19580934125966937, 0.34635680570840044, 0.19744386048733875, 0.25159355292175556, 0.07851239669421488, 0.20460942530261644, 0.14166666666666666, 0.6181139588443353, 0.2591858923416022, 0.7727420676242139, 0.0940813590969046, 0.3586253369272237, 0.17975487156521638, 0.36058901820124983, 0.3600424532251

### Old code 

In [46]:
# find indexes of df that correspond to df_train 
l = []
annotations = []
for keys, values in unique_KDs.items(): 
    for idx,i in enumerate(range(len(ds))): 
        score = sum(a == b for a,b in zip(values, ds['Statement Description'][i])) 
        if score >= round(min(1*len(values), 1*len(ds['Statement Description'][i]))): 
           # print(idx)
            l.append(keys)
            print(values)
            annotations.append([x for x in ds[i] if ds[i][str(x)] == 1])
            print([x for x in ds[i] if ds[i][str(x)] == 1])
print(len(l))

Knowledge of encryption algorithms
['1']
Knowledge of encryption algorithms
['1']
Knowledge of encryption algorithms
['1']
Knowledge of risk management processes
['7']
Knowledge of software engineering principles and practices
['2']
Knowledge of industry indicators
['0']
Knowledge of cyber operations principles and practices
['7']
Knowledge of operating system structures and internals
['7']
Knowledge of satellite-based communication systems and software
['0']
Knowledge of deployable forensics principles and practices
['1']
Knowledge of Payment Card Industry (PCI) data security standards and best practices
['0']
Knowledge of Personal Health Information (PHI) data security standards and best practices
['0']
Knowledge of operational planning processes
['7']
Knowledge of Risk Management Framework (RMF) requirements
['7']
Knowledge of language processing tools and techniques
['0']
Knowledge of data backup and recovery policies and procedures
['5']
Knowledge of database systems and software


In [18]:
# find out which of the KDs are new and which ones are not 
#print(unique_KDs)
new_KDs = pd.read_excel('data/NICE Framework Components v1.0.0(1).xlsx', sheet_name='v1.0.0 TKS Statements')
#print(new_KDs)

matching_keys = [key for key in unique_KDs.keys() if key in new_KDs['TKS Statement ID'].values]
non_matching_keys = [key for key in unique_KDs.keys() if key not in new_KDs['TKS Statement ID'].values]
print(len(matching_keys))
print(len(non_matching_keys))

non_matching_values = [value for key, value in unique_KDs.items() if key not in new_KDs['TKS Statement ID'].values]
print(len(non_matching_values))
print(non_matching_values)

# Quick method to get matching elements
def find_matching_elements(dictionary, df):
    # Find matching keys
    matching_keys = [key for key in dictionary.keys() if key in df['TKS Statement ID'].values]
    non_matching_keys = [key for key in dictionary.keys() if key not in df['TKS Statement ID'].values]
    # Find matching values - case insensitive and handles special characters
    matching_values = []
    non_matching_values = []
    for key, value in dictionary.items():
        # Convert both to lowercase for case-insensitive comparison and use regex=False
        if df['TKS Statement Description'].str.contains(value, case=False, regex=False, na=False).any():
            matching_values.append((key, value))
        else: 
            non_matching_values.append((key, value))
    
    return {
        'matching_keys': matching_keys,
        'matching_values': matching_values
    }

# Usage
results = find_matching_elements(unique_KDs, new_KDs);
print("Matching keys:", len(results['matching_keys']))
print("total keys: ", len(unique_KDs))
print(len(results['matching_values']))
# for the ones that are not new, load the KD labels 
#print(results)

536
36
36
['Knowledge of assessment remediation requirements', 'Knowledge of Business Impact Analysis (BIA)', 'Knowledge of change management processes', 'Knowledge of OT cybersecurity compliance requirements and best practices', 'Knowledge of control system environment risks, threats, and vulnerabilities', 'Knowledge of the Active Cyber Defense Cycle (ACDC)', 'Knowledge of active defense principles and practices', 'Knowledge of OT cybersecurity risk tolerance levels', 'Knowledge of Purdue Model levels', 'Knowledge of change management policies and procedures', 'Knowledge of OT cybersecurity inspection and testing policies and procedures', 'Knowledge of control system policies and procedures', 'Knowledge of OT safety systems', 'Knowledge of anomaly detection tools and techniques', 'Knowledge of change management processes', 'Knowledge of control system network architectures', 'Knowledge of cyber incidents impacting OT', 'Knowledge of industry hazards', 'Knowledge of life cycle manageme

In [40]:
# find indexes of df that correspond to df_train 
l = []
kd_values = list(unique_KDs.values())
for j in range(len(val_dataset)): 
    for idx,i in enumerate(range(len(unique_KDs))): #list(unique_KDs.values())
        score = sum(a == b for a,b in zip(val_dataset['Statement Description'][j], kd_values[i])) 
        if score >= round(min(1*len(val_dataset['Statement Description'][j]), 1*len(kd_values[i]))): 
           # print(idx)
            print('KD2017: ',i)
            print('description: ',val_dataset['Statement Description'][j])
            print('KD2025: ',idx)
            print('description: ',kd_values[i])
            l.append(idx)

KD2017:  5
description:  Knowledge of risk management processes (e.g., methods for assessing and mitigating risk)
KD2025:  5
description:  Knowledge of risk management processes
KD2017:  0
description:  Knowledge of encryption algorithms
KD2025:  0
description:  Knowledge of encryption algorithms
KD2017:  198
description:  Knowledge of data backup and recovery
KD2025:  198
description:  Knowledge of data backup and recovery policies and procedures
KD2017:  201
description:  Knowledge of database systems
KD2025:  201
description:  Knowledge of database systems and software
KD2017:  387
description:  Knowledge of digital rights management
KD2025:  387
description:  Knowledge of digital rights management (DRM) tools and techniques
KD2017:  330
description:  Knowledge of resiliency and redundancy
KD2025:  330
description:  Knowledge of resiliency and redundancy principles and practices
KD2017:  179
description:  Knowledge of Risk Management Framework (RMF) requirements
KD2025:  179
descrip